In [4]:
import datetime

import pandas as pd
from time import time

In [91]:
data = [
    ("La réunion est prévue le 15/04/2025 à 14h00.", True, 'fr'),
    ("Nous avons signé le contrat le 2 janvier 2022.", True, 'fr'),
    ("L'événement a été repoussé au March 12th, 2024.", True, 'fr'),
    ("Le colis est arrivé le 01-03-23, en avance.", True, 'fr'),
    ("2023-11-15 est la date de leur dernière connexion.", True, 'fr'),
    ("On se retrouve dimanche prochain, soit le 5 mai.", True, 'fr'),
    ("La conférence annuelle se tiendra en avril 2026.", True, 'fr'),
    ("Il a terminé son service en 1998.", True, 'fr'),
    ("Date limite : demain midi, pas plus tard.", True, 'fr'),
    ("Le spectacle est prévu dans trois semaines.", True, 'fr'),

    ("J'adore les balades en forêt après une journée de travail.", False, 'fr'),
    ("Cette série télé est absolument incroyable, tu devrais la voir !", False, 'fr'),
    ("Il est parti sans laisser de message.", False, 'fr'),
    ("Le serveur a crashé sans prévenir.", False, 'fr'),
    ("Elle prend toujours un café avant d’aller travailler.", False, 'fr'),

    ("Nous avons parlé de cela le premier vendredi de juin.", True, 'fr'),
    ("Il m’a dit de venir dans un mois pile.", True, 'fr'),
    ("Son anniversaire est le jour de la fête nationale.", True, 'fr'),
    ("Ils se sont rencontrés pendant l'été.", True, 'fr'),
    ("J’étais là le lendemain de Noël.", True, 'fr'),

    ("The meeting is scheduled for 04/15/2025 at 2 PM.", True, 'en'),
    ("We signed the contract on January 2nd, 2022.", True, 'en'),
    ("The event was postponed to March 12th, 2024.", True, 'en'),
    ("The package arrived on 03-01-23, earlier than expected.", True, 'en'),
    ("2023-11-15 is the date of their last login.", True, 'en'),
    ("We'll meet next Sunday, May 5th.", True, 'en'),
    ("The annual conference will be held in April 2026.", True, 'en'),
    ("He completed his service in 1998.", True, 'en'),
    ("Deadline: tomorrow at noon, no later.", True, 'en'),
    ("The show is planned for three weeks from now.", True, 'en'),

    ("I love walking in the woods after a long day at work.", False, 'en'),
    ("This TV show is amazing, you should watch it!", False, 'en'),
    ("He left without leaving a message.", False, 'en'),
    ("The server crashed without warning.", False, 'en'),
    ("She always grabs a coffee before going to work.", False, 'en'),

    ("We talked about it on the first Friday of June.", True, 'en'),
    ("He told me to come exactly in one month.", True, 'en'),
    ("Her birthday is on the national holiday.", True, 'en'),
    ("They met during the summer.", True, 'en'),
    ("I was there the day after Christmas.", True, 'en'),

    #plusieurs dates
    ("La réunion est prévue le 15/04/2025 et le 16/04/2025 à 14h00.", True, 'fr'),
    ("Nous avons signé le contrat le 2 janvier 2022 et le 3 janvier 2022.", True, 'fr'),
    ("L'événement a été repoussé au March 12th, 2024 et au March 13th, 2024.", True, 'fr'),
    ("Le colis est arrivé le 01-03-23 et le 02-03-23, en avance.", True, 'fr'),
    ("2023-11-15 et 2023-11-16 sont les dates de leur dernière connexion.", True, 'fr'),
    ("On se retrouve dimanche prochain, soit le 5 mai et le 6 mai.", True, 'fr'),
    ("La conférence annuelle se tiendra en avril 2026 et en mai 2026.", True, 'fr'),
    ("Il a terminé son service en 1998 et en 1999.", True, 'fr'),
    ("Date limite : demain midi et après-demain matin, pas plus tard.", True, 'fr'),
    ("Le spectacle est prévu dans trois semaines et dans quatre semaines.", True, 'fr'),
]

In [154]:
# Création du DataFrame
df = pd.DataFrame(data, columns=["text", "has_date", "lang"])
df_copy = df.copy()

In [141]:
def split_on_conjunctions(text):
    return re.split(r"\s+(et|and|,|;)\s+", text)

In [155]:
from preprocessing.text_cleaning import clean_text

df["text"] = df["text"].apply(clean_text)

In [143]:
df["text"]

0        [La réunion est prévue le 15/04/2025 à 14h00.]
1      [Nous avons signé le contrat le 2 janvier 2022.]
2     [L'événement a été repoussé au March 12th, 2024.]
3         [Le colis est arrivé le 01-03-23, en avance.]
4     [2023-11-15 est la date de leur dernière conne...
5     [On se retrouve dimanche prochain, soit le 5 m...
6     [La conférence annuelle se tiendra en avril 20...
7                   [Il a terminé son service en 1998.]
8            [Date limite: demain midi, pas plus tard.]
9         [Le spectacle est prévu dans trois semaines.]
10    [J'adore les balades en forêt après une journé...
11    [Cette série télé est absolument incroyable, t...
12              [Il est parti sans laisser de message.]
13                 [Le serveur a crashé sans prévenir.]
14    [Elle prend toujours un café avant d'aller tra...
15    [Nous avons parlé de cela le premier vendredi ...
16             [Il m'a dit de venir dans un mois pile.]
17    [Son anniversaire est le jour de la fête n

## Dateparser

In [144]:
from dateparser.search import search_dates
import re
from dateparser_data.settings import default_parsers

In [145]:
def dateparser_search_dates(text):
    """
    Utilise dateparser pour rechercher des dates dans le texte.
    """
    # Recherche de dates
    results = search_dates(text, languages=['fr', 'en'])

    return results

In [146]:
def contains_explicit_1_january(text: str) -> bool:
    patterns = [
        r"\b0?1[\/\-\. ]?0?1\b",  # 01/01, 1/1, 01-01, etc.
        r"\b(1er|1|01)[^\d]?(janvier|january)\b",  # 1 janvier, 1er janvier, 01 janvier
        r"\b(janvier|january)[^\d]*(1er|1|01)\b",  # janvier 1, january 1st
        r"\b(1st|first) of (january|janvier)\b"  # 1st of January
    ]
    for pattern in patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return True
    return False


def has_explicit_date_pattern(text):
    # Numérique : 15/08/2025, 15/08/25, 15-08-2025, 15-08-25
    numeric_patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",  # 15/08/25, 15/08/2025, 15-08-25, etc.
        r"\b\d{4}\b"  # année seule comme 2025
    ]

    # Texte : 15 avril 2025, 15 avril 25
    textual_patterns = [
        r"\b\d{1,2}\s+(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{2,4}\b"
    ]

    all_patterns = numeric_patterns + textual_patterns

    for pattern in all_patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return True
    return False


In [147]:
parsers = [parser for parser in default_parsers if parser != 'relative-time']

In [156]:
df_results = pd.DataFrame(
    columns=["text", "method", "time", "has_date", "result"],
)

In [157]:
for i, row in df.iterrows():
    start_time = time()
    lang = row["lang"]
    res_list = []

    try:
        # On divise le texte sur les conjonctions
        segments = [seg.strip() for seg in split_on_conjunctions(row["text"]) if seg.strip().lower() not in ["et", "and", ",", ";"] and seg.strip() != ""]

        for segment in segments:
            dates_found = search_dates(
                segment,
                languages=["fr", "en"],
                settings={
                    'PREFER_DAY_OF_MONTH': 'first',
                    'PREFER_MONTH_OF_YEAR': 'first',
                    'DATE_ORDER': 'DMY' if lang == 'fr' else 'MDY',
                    'PARSERS': parsers,
                }
            )

            if dates_found:
                for matched_text, result in dates_found:
                    res = {}

                    if result.day == 1 and result.month == 1:
                        if contains_explicit_1_january(matched_text):
                            res["year"] = result.year
                            res["month"] = result.month
                            res["day"] = result.day
                        else:
                            res["year"] = result.year
                            res["month"] = None
                            res["day"] = None
                    else:
                        res["year"] = result.year if has_explicit_date_pattern(matched_text) else None
                        res["month"] = result.month
                        res["day"] = result.day
                        res["hour"] = result.hour if result.hour else None
                        res["minute"] = result.minute if result.minute else None
                        res["second"] = result.second if result.second else None

                    res_list.append(res)
    except Exception as e:
        res_list = [{"error": str(e)}]

    elapsed_time = time() - start_time

    df_results = pd.concat([
        df_results,
        pd.DataFrame([{
            "text": row["text"],
            "method": "dateparser",
            "elapsed_time": elapsed_time,
            "expected_has_date": row["has_date"],
            "result": res_list
        }])
    ], ignore_index=True)


In [158]:

df_results

,text,method,time,has_date,result,elapsed_time,expected_has_date
0,La réunion est prévue le 15/04/2025 à 14h00.,dateparser,NaN,NaN,"[{'year': 2025, 'month': 4, 'day': 15, 'hour':...",0.003064,True
1,Nous avons signé le contrat le 2 janvier 2022.,dateparser,NaN,NaN,"[{'year': 2022, 'month': 1, 'day': 2, 'hour': ...",0.002208,True
2,"L'événement a été repoussé au March 12th, 2024.",dateparser,NaN,NaN,"[{'year': 2024, 'month': 1, 'day': 12, 'hour':...",0.001473,True
3,"Le colis est arrivé le 01-03-23, en avance.",dateparser,NaN,NaN,"[{'year': 2023, 'month': 3, 'day': 1, 'hour': ...",0.001053,True
4,2023-11-15 est la date de leur dernière connex...,dateparser,NaN,NaN,[],0.000839,True
5,"On se retrouve dimanche prochain, soit le 5 mai.",dateparser,NaN,NaN,"[{'year': None, 'month': 1, 'day': 20, 'hour':...",0.002328,True
6,La conférence annuelle se tiendra en avril 2026.,dateparser,NaN,NaN,"[{'year': 2026, 'month': 4, 'day': 1, 'hour': ...",0.001016,True
7,Il a terminé son service en 1998.,dateparser,NaN,NaN,"[{'year': 1998, 'month': None, 'day': None}]",0.000847,True
8,"Date limite: demain midi, pas plus tard.",dateparser,NaN,NaN,[],0.001791,True
9,Le spectacle est prévu dans trois semaines.,dateparser,NaN,NaN,[],0.001146,True
